# Model Training & Benchmarking: Short-Term Glucose Spike Forecaster
### Happiest Health Digital Twin Challenge 2026

In this notebook, we develop and benchmark machine learning models for forecasting **acute glucose spikes (< 2 hours)**:
$$\text{Target } y(t) = 1 \quad \text{if } \max_{s \in (t, t + 120\text{m}]} G(s) \ge 180\text{ mg/dL} \text{ or } \Delta G \ge 50\text{ mg/dL}$$

### Models Compared:
1. **Logistic Regression (L2 Baseline)**
2. **Random Forest Classifier (Ensemble Baseline)**
3. **XGBoost / Gradient Boosted Trees (Primary Model)**

**Strict Chronological Split**: 70% Train, 15% Validation, 15% Test. Zero future leakage.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.loader import load_patient_data
from src.features.engineer import build_feature_matrix, create_temporal_splits, FEATURE_COLUMNS, TARGET_COLUMN
from src.models.train import train_and_compare_models

# 1. Load data & engineer features
patient_profile, stream_df = load_patient_data(patient_id="PT-SYNTH-001")
df_features = build_feature_matrix(stream_df, patient_profile=patient_profile, include_target=True)
df_train, df_val, df_test = create_temporal_splits(df_features)

print(f"Feature matrix shape: {df_features.shape}")
print(f"Train set: {len(df_train)} | Val set: {len(df_val)} | Test set: {len(df_test)}")
print(f"Spike prevalence: {df_train[TARGET_COLUMN].mean()*100:.1f}%")

## 2. Train and Benchmark Models

In [ ]:
summary = train_and_compare_models(df_train, df_val, df_test, output_dir="models")
print("Benchmarking complete. Best model:", summary["best_model_selected"])
benchmark = pd.DataFrame(summary["test_benchmark_results"]).T
benchmark[['roc_auc', 'pr_auc', 'accuracy', 'precision', 'recall_sensitivity', 'specificity', 'f1_score']]

## 3. Explainability & SHAP Feature Attributions
Inspecting which physiological signals drive the model's predictions.

In [ ]:
from src.models.explain import ModelExplainer
import joblib

pkg = joblib.load("models/deployed_model_package.joblib")
model = pkg["model"]
explainer = ModelExplainer(model=model, feature_names=FEATURE_COLUMNS, background_data=df_train)

# Explain an acute spike scenario
spike_sample = df_test[df_test[TARGET_COLUMN] == 1].iloc[[0]]
explanations = explainer.explain_instance(spike_sample, top_k=5)

print("Top Driving Factors for Sample Spike:")
for exp in explanations:
    print(f" - {exp['label']}: {exp['contribution']} ({exp['direction']})")